# kin_00 — Movement QC: noise vs. real tongue movement

Two noise signals, examined in order of confidence:

1. **Duration / frame-count** (sections 3-4) — `duration==0` / `n_datapoints==1` and
   sub-physical-floor movements are unreliable by construction: below the 50 Hz
   Butterworth cutoff (10 frames / 20 ms), the velocity estimate is dominated by
   tracking jitter. No GMM — just distributions and a physically-grounded floor.

2. **Spatial radius from home** (sections 5-6) — a movement whose tongue keypoint
   strays far beyond the session's typical tongue cloud is likely a misidentified
   body part. Ceiling chosen by inspecting the pooled distribution, confirmed by
   trajectory plots and video frames on Code Ocean.

**Diagnostic only.** Nothing is persisted; `kin_01` is not modified.

**Local vs Code Ocean**
- Local: sections 3–5 render (radius from aggregate endpoint proxy); trajectory grid
  and video frames print skip messages.
- CO: all sections run with per-frame `tongue_kins.parquet`.

## 1. Setup

In [ ]:
%matplotlib inline
from pathlib import Path
import subprocess
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from plotstyle import apply_style, PALETTE, style_ax, save_fig
apply_style()

In [ ]:
if Path("/root/capsule").exists():
    ENV       = "codeocean"
    SCRATCH   = Path("/root/capsule/scratch")
    FOR_LOCAL = SCRATCH / "for_local"
else:
    ENV       = "local"
    FOR_LOCAL = Path("/Users/mib/Documents/Code/kinematics_analysis/data/for_local")
    SCRATCH   = FOR_LOCAL.parent

FIG_DIR  = SCRATCH / "figures" / "kin_00_movement_qc"
SAVE_FIG = False
print(f"ENV={ENV}")

## 2. Load data

Full, **unfiltered** dataset — evaluating whether a filter is justified, so
`kin_01`'s `out_duration > 0.05` quality filter is not applied here.

In [ ]:
if ENV == "codeocean":
    movements_path = SCRATCH / "all_tongue_movements_04022026" / "all_tongue_movements_04022026.parquet"
else:
    movements_path = FOR_LOCAL / "all_tongue_movements_04022026.parquet"

all_tongue_movements = pd.read_parquet(movements_path)
print("Shape:", all_tongue_movements.shape)
print("Sessions:", all_tongue_movements["session"].nunique())

## 3. Duration and frame-count distributions

The 50 Hz Butterworth low-pass filter applied before velocity differentiation
sets a hard temporal resolution floor of 1/50 s = **20 ms = 10 frames** at
500 Hz. Below that, velocity/duration estimates reflect filter smoothing, not
kinematics. `duration==0` / `n_datapoints==1` are a special sub-case: a single
tracked frame with no velocity estimate at all.

In [ ]:
PHYSICAL_FLOOR_FRAMES = 10      # 50 Hz cutoff -> 20 ms = 10 frames @ 500 Hz
PHYSICAL_FLOOR_S      = PHYSICAL_FLOOR_FRAMES / 500.0


def _qc_hist(ax, vals, label, log=False, color=None, vlines=None):
    color = color or PALETTE["neutral"]
    vals = np.asarray(vals, dtype=float)
    vals = vals[~np.isnan(vals)]
    if log:
        vals = vals[vals > 0]
        vals = np.log10(vals)
        label = f"log10({label})"
    lo, hi = np.nanpercentile(vals, 1), np.nanpercentile(vals, 99)
    vals_c = vals[(vals >= lo) & (vals <= hi)]
    ax.hist(vals_c, bins=60, color=color, edgecolor="white", linewidth=0.3)
    med = float(np.median(vals))
    ax.axvline(med, color=PALETTE["neg"], lw=1.2, ls="--", label=f"median={med:.3g}")
    if vlines:
        for v_val, v_label, v_color in vlines:
            x = (np.log10(v_val) if (log and v_val > 0) else v_val)
            ax.axvline(x, color=v_color, lw=1.2, ls=":", label=v_label)
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
    ax.legend(fontsize=7)
    style_ax(ax)


dur  = all_tongue_movements["duration"].to_numpy()
ndp  = all_tongue_movements["n_datapoints"].to_numpy()
dfp  = all_tongue_movements["dropped_frames_pct"].to_numpy()
pv   = all_tongue_movements["peak_velocity"].to_numpy()
td   = all_tongue_movements["total_distance"].to_numpy()

n_zero_dur = int((dur == 0).sum())
n_single   = int((ndp == 1).sum())
n_nan_td   = int(np.isnan(td).sum())

floor_vlines_s  = [(PHYSICAL_FLOOR_S, f"10-frame floor ({PHYSICAL_FLOOR_S*1000:.0f} ms)", PALETTE["accent"])]
floor_vlines_fr = [(PHYSICAL_FLOOR_FRAMES, f"floor = {PHYSICAL_FLOOR_FRAMES} frames", PALETTE["accent"])]

fig, axes = plt.subplots(2, 5, figsize=(22, 7))

_qc_hist(axes[0, 0], dur, "Duration (s)", vlines=floor_vlines_s)
_qc_hist(axes[1, 0], dur, "Duration (s)", log=True, vlines=floor_vlines_s)
axes[1, 0].set_title(f"n={n_zero_dur:,} zero-duration excluded from log panel", fontsize=7)

_qc_hist(axes[0, 1], ndp, "N frames", vlines=floor_vlines_fr)
_qc_hist(axes[1, 1], ndp, "N frames", log=True, vlines=floor_vlines_fr)
axes[0, 1].set_title(f"n={n_single:,} single-frame detections", fontsize=7)

_qc_hist(axes[0, 2], dfp, "Dropped frames (%)")
axes[1, 2].set_visible(False)
pct_zero_drop = 100 * float(np.nanmean(dfp == 0))
axes[0, 2].set_title(f"{pct_zero_drop:.1f}% have 0 dropped frames", fontsize=7)

_qc_hist(axes[0, 3], pv, "Peak velocity (a.u.)")
_qc_hist(axes[1, 3], pv, "Peak velocity (a.u.)", log=True)

_qc_hist(axes[0, 4], td, "Total distance (a.u.)")
_qc_hist(axes[1, 4], td, "Total distance (a.u.)", log=True)
axes[1, 4].set_title(f"n={n_nan_td:,} NaN (no trajectory) excluded from log panel", fontsize=7)

fig.suptitle(
    f"Whole-movement QC  (n={len(all_tongue_movements):,}, unfiltered) — "
    f"dotted line = 10-frame / 20 ms physical floor",
    fontsize=10,
)
plt.tight_layout()
save_fig(fig, "qc_univariate_distributions", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

n_below = int((ndp < PHYSICAL_FLOOR_FRAMES).sum())
print(f"Below floor (< {PHYSICAL_FLOOR_FRAMES} frames):  n={n_below:,}  ({n_below/len(all_tongue_movements)*100:.1f}%)")
print(f"Single-frame (n_datapoints==1):     n={n_single:,}  ({n_single/len(all_tongue_movements)*100:.1f}%)")
print(f"Zero-duration (duration==0):        n={n_zero_dur:,}  ({n_zero_dur/len(all_tongue_movements)*100:.1f}%)")

## 4. Velocity-distance 2-frame degeneracy

With only 2 frames there is exactly one displacement vector, so `peak_velocity`
and `total_distance` are forced onto the deterministic line `v = 500 × d` (one
sample = max = mean = total). With more frames the two metrics decouple. The
rail of 2-frame detections along this diagonal confirms their velocity estimate
is meaningless.

In [ ]:
sub = all_tongue_movements.dropna(subset=["total_distance", "peak_velocity"]).copy()
log_d = np.log10(sub["total_distance"].to_numpy())
log_v = np.log10(sub["peak_velocity"].to_numpy())

fig, ax = plt.subplots(figsize=(7, 5.5))
sc = ax.scatter(
    log_d, log_v,
    s=3, alpha=0.35,
    c=np.log10(sub["n_datapoints"].to_numpy()),
    cmap="viridis", rasterized=True,
)
cb = fig.colorbar(sc, ax=ax)
cb.set_label("log10(n_frames)")

d_ref = np.linspace(log_d.min(), log_d.max(), 50)
ax.plot(d_ref, d_ref + np.log10(500), color=PALETTE["neg"], ls="--", lw=1.2,
        label="v = 500 × d  (forced, n=2)")

ax.set_xlabel("log10(total distance)")
ax.set_ylabel("log10(peak velocity)")
ax.set_title("Velocity-distance colored by frame count\n"
             "2-frame detections sit on the forced diagonal")
ax.legend(fontsize=7)
style_ax(ax)
plt.tight_layout()
save_fig(fig, "velocity_distance_by_n_datapoints", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

## 5. Spatial radius from home (tongue-only)

For each movement, compute the **maximum** distance its tongue keypoint reached
from the session-median tongue location ("home"). A movement whose tongue never
leaves the cloud of typical positions is unlikely to be a misidentified body
part; one that strays far is a candidate for misdetection.

- **`r_max`** — farthest excursion (the key metric).
- **`r_med`** — median distance-to-home; cross-check. `r_max` and `r_med` agree
  for wholly-displaced detections (always far), but diverge on single-frame
  glitches (one distant frame among many normal ones).
- Normalize per session: `r_max_norm = r_max / session_median(r_max)` so that
  sessions with different pixel scales are pooled fairly.

**Code Ocean**: per-frame positions from `intermediate_data/tongue_kins.parquet`.  
**Local**: `r_max` approximated from aggregate `endpoint_x/y` vs session-median
endpoint — preview only; underestimates true excursion on movements that
turn before the endpoint.

In [ ]:
if ENV == "codeocean":
    base_dirs = [SCRATCH / "session_analysis_mlk"]
    sess_dirs = {}
    for bd in base_dirs:
        if bd.is_dir():
            for sp in bd.iterdir():
                kpath = sp / "intermediate_data" / "tongue_kins.parquet"
                if sp.is_dir() and kpath.exists():
                    sess_dirs[sp.name] = sp
    print(f"Found tongue_kins for {len(sess_dirs)} session directories.")

    radius_rows = []
    tongue_cloud_rows = []
    for sess_name in sorted(sess_dirs):
        sp = sess_dirs[sess_name]
        try:
            kins = pd.read_parquet(sp / "intermediate_data" / "tongue_kins.parquet")
        except Exception as e:
            print(f"[skip] {sess_name}: {e}")
            continue

        ok = kins["x"].notna() & kins["y"].notna()
        home_x = float(kins.loc[ok, "x"].median())
        home_y = float(kins.loc[ok, "y"].median())
        kins["_r"] = np.hypot(kins["x"] - home_x, kins["y"] - home_y)

        with_mid = kins.dropna(subset=["movement_id"])
        mov = (
            with_mid.groupby("movement_id")["_r"]
            .agg(["max", "median"])
            .reset_index()
        )
        mov.columns = ["movement_id", "r_max", "r_med"]
        mov["session"] = sess_name
        mov["home_x"]  = home_x
        mov["home_y"]  = home_y
        radius_rows.append(mov)

        # Tongue-position cloud sample (≤ 5000 frames per session) for cloud plot
        cloud_samp = kins.loc[ok, ["x", "y"]].sample(
            min(5000, ok.sum()), random_state=42
        ).copy()
        cloud_samp["session"] = sess_name
        cloud_samp["home_x"]  = home_x
        cloud_samp["home_y"]  = home_y
        tongue_cloud_rows.append(cloud_samp)

    df_radius   = pd.concat(radius_rows, ignore_index=True)      if radius_rows      else pd.DataFrame()
    tongue_cloud = pd.concat(tongue_cloud_rows, ignore_index=True) if tongue_cloud_rows else pd.DataFrame()
    df_rad = all_tongue_movements.merge(df_radius, on=["session", "movement_id"], how="left")
    print(f"r_max matched: {df_rad['r_max'].notna().sum():,} / {len(df_rad):,} movements")

else:
    sess_dirs    = {}
    tongue_cloud = None
    df_rad = all_tongue_movements.copy()
    home_stats = (
        df_rad.groupby("session")[["endpoint_x", "endpoint_y"]]
        .median()
        .rename(columns={"endpoint_x": "home_x", "endpoint_y": "home_y"})
    )
    df_rad = df_rad.join(home_stats, on="session")
    df_rad["r_max"] = np.hypot(
        df_rad["endpoint_x"] - df_rad["home_x"],
        df_rad["endpoint_y"] - df_rad["home_y"],
    )
    df_rad["r_med"] = df_rad["r_max"]
    print("Local: r_max approximated from endpoint distance to session-median endpoint.")
    print("  Per-frame precision and 2D tongue cloud require Code Ocean.")

# Per-session normalization
s_med = df_rad.groupby("session")["r_max"].transform("median")
df_rad["r_max_norm"] = df_rad["r_max"] / (s_med + 1e-9)
df_rad["r_med_norm"] = df_rad["r_med"] / (s_med + 1e-9)

valid_r = df_rad["r_max"].notna() & (df_rad["r_max"] >= 0)
print(f"\nr_max summary ({valid_r.sum():,} movements with valid r_max):")
print(df_rad.loc[valid_r, ["r_max", "r_max_norm"]].describe().round(2).to_string())

In [ ]:
# Per-session 2D tongue position cloud with home marked and candidate radius circles.
# Code Ocean only — requires per-frame tongue_kins.parquet.

if ENV != "codeocean" or tongue_cloud is None or tongue_cloud.empty:
    print("Per-session tongue cloud: Code Ocean only (requires per-frame tongue_kins.parquet).")
else:
    sessions_to_plot = sorted(tongue_cloud["session"].unique())[:6]
    ncols = 3
    nrows = (len(sessions_to_plot) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 4))
    axes = np.array(axes).ravel()

    for ax_i, sess in enumerate(sessions_to_plot):
        ax = axes[ax_i]
        cloud = tongue_cloud[tongue_cloud["session"] == sess]
        home_x = float(cloud["home_x"].iloc[0])
        home_y = float(cloud["home_y"].iloc[0])
        sess_r_med = float(df_rad.loc[df_rad["session"] == sess, "r_max"].median())

        ax.scatter(cloud["x"], cloud["y"], s=1, alpha=0.3,
                   color=PALETTE["neutral"], rasterized=True)
        ax.plot(home_x, home_y, "x", color=PALETTE["neg"], ms=8, mew=2, label="home")

        for mult, color, ls in [
            (1, PALETTE["pos"],    "--"),
            (3, PALETTE["sig"],    "-."),
            (5, PALETTE["accent"], ":"),
        ]:
            circ = mpatches.Circle(
                (home_x, home_y), mult * sess_r_med,
                fill=False, edgecolor=color, lw=1.2, linestyle=ls,
                label=f"{mult}× median r_max",
            )
            ax.add_patch(circ)

        ax.set_aspect("equal")
        ax.set_title(sess[:24], fontsize=7)
        ax.legend(fontsize=6)
        style_ax(ax)

    for ax in axes[len(sessions_to_plot):]:
        ax.set_visible(False)

    fig.suptitle("Per-session tongue cloud with candidate radius ceilings\n"
                 "(1× / 3× / 5× session-median r_max)")
    plt.tight_layout()
    save_fig(fig, "tongue_cloud_sessions", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()

In [ ]:
# Pooled distribution of normalized r_max (log x-axis).
# r_max and r_med agree for wholly-displaced detections (both are high);
# they diverge on single-frame glitches (r_max high, r_med low — one
# displaced frame among otherwise normal positions).
# The user reads off the ceiling from the bulk+tail structure.

if ENV == "local":
    print("[Local preview] r_max from aggregate endpoints — tail shape informative "
          "but underestimates true per-frame max excursion. Run on Code Ocean for precision.\n")

valid_r = df_rad["r_max_norm"].notna() & (df_rad["r_max_norm"] > 0)
r_max_vals = df_rad.loc[valid_r, "r_max_norm"].to_numpy()
r_med_vals = df_rad.loc[valid_r, "r_med_norm"].to_numpy()
log_rmax = np.log10(r_max_vals)
log_rmed = np.log10(r_med_vals[r_med_vals > 0])

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(log_rmax, bins=80, color=PALETTE["neutral"],
        edgecolor="white", linewidth=0.3, alpha=0.8, label="r_max (peak excursion)")
ax.hist(log_rmed, bins=80, color=PALETTE["pos"],
        edgecolor="white", linewidth=0.3, alpha=0.4, label="r_med (median dist-to-home)")

# Candidate ceiling lines
for mult, color, ls in [
    (3,  PALETTE["sig"],    "-."),
    (5,  PALETTE["neg"],    "--"),
    (10, PALETTE["accent"], ":"),
]:
    pct_below = 100 * float(np.mean(r_max_vals < mult))
    ax.axvline(np.log10(mult), color=color, lw=1.3, ls=ls,
               label=f"ceiling {mult}×  (keeps {pct_below:.0f}% of movements)")

# Percentile table
print("r_max_norm percentiles:")
for p in (90, 95, 99, 99.5):
    v = float(np.percentile(r_max_vals, p))
    print(f"  {p:5.1f}th pctile:  {v:.2f}×  (log10 = {np.log10(v):.2f})")

# Secondary x axis in raw multiples
x_raw = [0.2, 0.5, 1, 2, 5, 10, 20, 50]
xlim = ax.get_xlim()
valid_ticks = [(t, np.log10(t)) for t in x_raw if xlim[0] <= np.log10(t) <= xlim[1]]
if valid_ticks:
    ax2 = ax.twiny()
    ax2.set_xlim(xlim)
    ax2.set_xticks([v for _, v in valid_ticks])
    ax2.set_xticklabels([f"{t}×" for t, _ in valid_ticks], fontsize=7)
    ax2.set_xlabel("r_max / session-median r_max")

ax.set_xlabel("log10(r_max / session-median r_max)")
ax.set_ylabel("Count")
ax.set_title("Pooled distribution of normalized radius-from-home\n"
             "r_max = peak excursion; r_med = cross-check (diverges on single-frame glitches)")
ax.legend(fontsize=8)
style_ax(ax)
plt.tight_layout()
save_fig(fig, "r_max_norm_distribution", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

In [ ]:
# r_max vs duration: are spatial outliers and short detections distinct
# populations, or do they largely overlap?
# If orthogonal — each catches noise the other misses; both criteria needed.
# If overlapping — a single combined criterion is sufficient.

has_r = df_rad["r_max_norm"].notna()
dur_v = df_rad.loc[has_r, "duration"].to_numpy()
r_v   = df_rad.loc[has_r, "r_max_norm"].to_numpy()
ndp_v = df_rad.loc[has_r, "n_datapoints"].to_numpy()

# Push zero-duration to just below the floor for log plotting (avoids log10(0) warning)
log_dur = np.full(len(dur_v), np.log10(0.5 / 500))
pos_mask = dur_v > 0
log_dur[pos_mask] = np.log10(dur_v[pos_mask])
log_r = np.log10(np.maximum(r_v, 1e-3))

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(
    log_dur, log_r,
    s=2, alpha=0.2,
    c=np.log10(ndp_v),
    cmap="viridis", rasterized=True,
)
cb = fig.colorbar(sc, ax=ax)
cb.set_label("log10(n_frames)")

ax.axvline(np.log10(PHYSICAL_FLOOR_S), color=PALETTE["accent"], lw=1.2, ls=":",
           label=f"10-frame floor ({PHYSICAL_FLOOR_S*1000:.0f} ms)")
ax.axhline(np.log10(5), color=PALETTE["neg"], lw=1.2, ls="--",
           label="r_max_norm = 5×  (candidate ceiling)")

ax.set_xlabel("log10(duration)")
ax.set_ylabel("log10(r_max / session median)")
ax.set_title("Spatial outliers vs. short detections — same movements?")
ax.legend(fontsize=8)
style_ax(ax)
plt.tight_layout()
save_fig(fig, "r_max_vs_duration", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

## 6. Trajectory plots (Code Ocean only)

Plot each movement's per-frame `(x, y)` path on top of the session tongue
cloud, home marked. Grid: N normal movements (low `r_max_norm`, ≥ 10 frames)
vs N high-`r_max_norm` vs N single-frame. Visual comparison lets you see
whether high-radius detections trace an implausible path or just a long lick.

In [ ]:
if ENV != "codeocean" or not sess_dirs:
    print("Trajectory plots: Code Ocean only (requires per-frame tongue_kins.parquet).")
else:
    # Pick the session with the most extreme high-r_max detections for a clear contrast
    session_r99 = df_rad.groupby("session")["r_max_norm"].quantile(0.99)
    demo_sess   = session_r99.idxmax()
    sp          = sess_dirs.get(demo_sess)

    if sp is None:
        print(f"No tongue_kins found for session {demo_sess}")
    else:
        kins = pd.read_parquet(sp / "intermediate_data" / "tongue_kins.parquet")
        sess_movs = df_rad[df_rad["session"] == demo_sess].dropna(subset=["r_max_norm"])

        home_x = float(kins["x"].median())
        home_y = float(kins["y"].median())

        N   = 6
        rng = np.random.default_rng(42)

        normal_ids = sess_movs[
            (sess_movs["r_max_norm"] < 2) & (sess_movs["n_datapoints"] >= PHYSICAL_FLOOR_FRAMES)
        ]["movement_id"].to_numpy()
        high_r_ids = sess_movs[sess_movs["r_max_norm"] >= 5]["movement_id"].to_numpy()
        single_ids = sess_movs[sess_movs["n_datapoints"] == 1]["movement_id"].to_numpy()

        def _sample(ids, n):
            if len(ids) == 0:
                return []
            return list(rng.choice(ids, size=min(n, len(ids)), replace=False))

        normal_ids = _sample(normal_ids, N)
        high_r_ids = _sample(high_r_ids, N)
        single_ids = _sample(single_ids, N)

        kins_by_mid = kins.dropna(subset=["movement_id"]).groupby("movement_id")

        categories = [
            ("Normal  (r_max_norm < 2×, ≥ 10 frames)", normal_ids, PALETTE["pos"]),
            ("High radius  (r_max_norm ≥ 5×)",         high_r_ids, PALETTE["neg"]),
            ("Single-frame",                            single_ids, PALETTE["accent"]),
        ]

        fig, axes = plt.subplots(3, N, figsize=(N * 2.5, 9))
        cloud = tongue_cloud[tongue_cloud["session"] == demo_sess]

        for row_idx, (cat_label, ids, color) in enumerate(categories):
            for col_idx in range(N):
                ax = axes[row_idx, col_idx]
                ax.scatter(cloud["x"], cloud["y"], s=0.5, alpha=0.1,
                           color=PALETTE["neutral"], rasterized=True)
                ax.plot(home_x, home_y, "x", color="black", ms=6, mew=1.5)

                if col_idx < len(ids):
                    mid = ids[col_idx]
                    if mid in kins_by_mid.groups:
                        path = (
                            kins_by_mid.get_group(mid)[["x", "y"]]
                            .dropna()
                        )
                        if len(path) >= 1:
                            ax.plot(path["x"], path["y"],
                                    color=color, lw=1.2, alpha=0.85)
                            ax.scatter(path["x"].iloc[0], path["y"].iloc[0],
                                       s=20, color="green", zorder=3)
                            ax.scatter(path["x"].iloc[-1], path["y"].iloc[-1],
                                       s=20, color="red", zorder=3)
                else:
                    ax.set_visible(False)

                ax.set_aspect("equal")
                style_ax(ax)
                ax.set_xticks([])
                ax.set_yticks([])
                if col_idx == 0:
                    ax.set_ylabel(cat_label, fontsize=8)

        fig.suptitle(
            f"Tongue trajectories — {demo_sess[:36]}\n"
            f"Green = start, red = end, grey cloud = session tongue positions",
            fontsize=9,
        )
        plt.tight_layout()
        save_fig(fig, "trajectory_grid", fig_dir=FIG_DIR, save=SAVE_FIG)
        plt.show()

## 7. Video-frame ground-truthing (Code Ocean only)

Extract individual labeled-video frames at the exact timestamps of flagged
movements and display them for visual inspection. Confirms (or refutes) that
a high-radius or single-frame detection is a tongue vs. another body part.

**Pattern** (from `batch_clips.ipynb` cell `a23ebc74`):
```
ffmpeg -hide_banner -loglevel error -ss {t} -i {video} -frames:v 1 -q:v 2 -y {out.png}
```
`tongue_kins.time` = seconds from video start = the value for ffmpeg `-ss`.

In [ ]:
from IPython.display import Image, display as _display

PRED_CSV_JSON = Path("/root/capsule/scratch/pred_csv_list_20250113.json")
_SCHEMA_SHOWN = [False]  # mutable flag (avoids global)


def resolve_labeled_video(session_name):
    """Return the labeled-video Path for a session, or None if not found."""
    if PRED_CSV_JSON.exists():
        with open(PRED_CSV_JSON) as fh:
            entries = json.load(fh)
        if not _SCHEMA_SHOWN[0]:
            _SCHEMA_SHOWN[0] = True
            print(f"JSON: {len(entries)} entries")
            if entries:
                sample = entries[0]
                print(f"  keys:   {list(sample.keys()) if isinstance(sample, dict) else type(sample)}")
                print(f"  sample: {sample}")
        for entry in entries:
            if isinstance(entry, dict):
                for key, val in entry.items():
                    if session_name in str(val) and "videoprocessed" in str(val):
                        vpath = (
                            Path(str(val))
                            / "pred_outputs" / "video_preds"
                            / "labeled_videos" / "video_labeled.mp4"
                        )
                        if vpath.exists():
                            return vpath
            elif session_name in str(entry):
                # Entry may be a path string
                for cand in Path("/root/capsule/data").glob(
                    f"{session_name}_videoprocessed_*/pred_outputs/"
                    "video_preds/labeled_videos/video_labeled.mp4"
                ):
                    if cand.exists():
                        return cand

    # Fallback: glob directly under data
    for cand in Path("/root/capsule/data").glob(
        f"{session_name}_videoprocessed_*/pred_outputs/"
        "video_preds/labeled_videos/video_labeled.mp4"
    ):
        if cand.exists():
            return cand
    return None


def show_movement_frames(session_name, movement_id, n=6):
    """Extract and display n frames spanning a movement's time range."""
    if ENV != "codeocean":
        print("show_movement_frames: Code Ocean only.")
        return

    sp = sess_dirs.get(session_name)
    if sp is None:
        print(f"No session dir found for {session_name}")
        return

    kins = pd.read_parquet(sp / "intermediate_data" / "tongue_kins.parquet")
    mov_frames = (
        kins[kins["movement_id"] == movement_id]
        .dropna(subset=["time"])
        .sort_values("time")
    )
    if mov_frames.empty:
        print(f"No frames for movement_id={movement_id}")
        return

    times = mov_frames["time"].to_numpy()
    indices = np.linspace(0, len(times) - 1, min(n, len(times)), dtype=int)
    selected = times[indices]

    video = resolve_labeled_video(session_name)
    if video is None:
        print(f"Could not resolve labeled video for {session_name}")
        return
    print(f"Video: {video}")

    out_dir = sp / "intermediate_data"
    print(f"movement_id={movement_id}  ({len(times)} frames, "
          f"t={times[0]:.3f}–{times[-1]:.3f}s)")

    for i, t in enumerate(selected):
        out_png = out_dir / f"_frame_{movement_id}_{t:.3f}s.png"
        subprocess.run(
            ["ffmpeg", "-hide_banner", "-loglevel", "error",
             "-ss", str(t), "-i", str(video),
             "-frames:v", "1", "-q:v", "2", "-y", str(out_png)],
            check=True,
        )
        print(f"  frame {i+1}/{len(selected)}: t={t:.3f}s")
        _display(Image(filename=str(out_png)))

In [ ]:
# Smoke-test: resolve one session's video path and print it.
# Then show frames for a few high-r_max and single-frame movements.

if ENV != "codeocean" or not sess_dirs:
    print("Video ground-truthing: Code Ocean only.")
else:
    demo_sess_r = df_rad.groupby("session")["r_max_norm"].quantile(0.99).idxmax()
    print(f"Demo session: {demo_sess_r}")
    video_path = resolve_labeled_video(demo_sess_r)
    print(f"Resolved video: {video_path}")

    sess_movs = df_rad[df_rad["session"] == demo_sess_r].dropna(subset=["r_max_norm"])

    print("\n=== High r_max movements ===")
    high_r = sess_movs.nlargest(3, "r_max_norm")[
        ["movement_id", "r_max_norm", "duration", "n_datapoints"]
    ]
    print(high_r.to_string(index=False))
    for mid in high_r["movement_id"].tolist():
        show_movement_frames(demo_sess_r, mid, n=4)

    print("\n=== Single-frame movements ===")
    single = sess_movs[sess_movs["n_datapoints"] == 1]
    sample_single = single.sample(
        min(3, len(single)), random_state=42
    ) if len(single) > 0 else pd.DataFrame()
    for mid in sample_single["movement_id"].tolist():
        show_movement_frames(demo_sess_r, mid, n=1)

## 8. Synthesis and recommendation

**Candidate noise criteria** (set `R_CEILING` after inspecting the distribution
in section 5 and the video frames in section 7):

```
is_candidate_noise  =  (duration == 0)
                    OR (n_datapoints < 10)       # below physical floor
                    OR (r_max_norm > R_CEILING)  # tongue strays too far from home
```

**Compared to `kin_01`'s current `out_duration > 0.05` filter** — the synthesis
cell below reports how many movements each criterion removes and how much they
overlap. If the current filter over-removes real movements (because
`out_duration` is shortened by outbound truncation), the whole-movement
criterion retains more signal.

**Nothing is applied here.** This notebook is diagnostic only.

In [ ]:
# Set R_CEILING to the value you read off the r_max_norm distribution above.
# Leave as None to skip the spatial filter from the synthesis.
R_CEILING = None   # e.g. 5.0

is_zero_dur  = df_rad["duration"] == 0
is_sub_floor = df_rad["n_datapoints"] < PHYSICAL_FLOOR_FRAMES
is_short     = is_zero_dur | is_sub_floor

print("Short / sub-resolution noise candidates:")
print(f"  zero-duration:               n={is_zero_dur.sum():>7,}  ({is_zero_dur.mean()*100:.1f}%)")
print(f"  sub-floor (n_frames < {PHYSICAL_FLOOR_FRAMES}):   n={is_sub_floor.sum():>7,}  ({is_sub_floor.mean()*100:.1f}%)")
print(f"  combined (short):            n={is_short.sum():>7,}  ({is_short.mean()*100:.1f}%)")

if R_CEILING is not None:
    is_displaced = df_rad["r_max_norm"] > R_CEILING
    is_displaced = is_displaced.fillna(False)
    is_candidate = is_short | is_displaced
    overlap = (is_short & is_displaced).sum()
    print(f"\nSpatial noise (r_max_norm > {R_CEILING}×):")
    print(f"  displaced:                   n={is_displaced.sum():>7,}  ({is_displaced.mean()*100:.1f}%)")
    print(f"  overlap with short:          n={overlap:>7,}")
    print(f"  spatial-only (not short):    n={(is_displaced & ~is_short).sum():>7,}")
    print(f"  combined candidate noise:    n={is_candidate.sum():>7,}  ({is_candidate.mean()*100:.1f}%)")
else:
    is_candidate = is_short
    print("\n(R_CEILING not set — spatial criterion not included in synthesis.)")

# Compare to kin_01's current filter
has_out = df_rad["out_duration"].notna()
current_keeps   = has_out & (df_rad["out_duration"] > 0.05)
current_removes = ~current_keeps
new_removes     = is_candidate

agree_remove    = (current_removes & new_removes).sum()
current_only    = (current_removes & ~new_removes).sum()
new_only        = (~current_removes & new_removes).sum()

print(f"\n--- Comparison to kin_01 out_duration > 0.05 ---")
print(f"  kin_01 removes:              n={current_removes.sum():>7,}  ({current_removes.mean()*100:.1f}%)")
print(f"  candidate criterion removes: n={new_removes.sum():>7,}  ({new_removes.mean()*100:.1f}%)")
print(f"  both remove:                 n={agree_remove:>7,}")
print(f"  kin_01 removes, candidate keeps (over-removal): n={current_only:>7,}")
print(f"  candidate removes, kin_01 keeps (missed noise): n={new_only:>7,}")